In [1]:
# pip install transformers jiwer
import os 
import numpy as np 
from tqdm.notebook import tqdm # for displaying process bar
import torch 
import torch.nn as nn 
from torch import optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T 
import torchaudio.functional as F 
import matplotlib.pyplot as plt 
from transformers import Wav2Vec2CTCTokenizer, get_cosine_schedule_with_warmup 
from jiwer import wer # to evaluate the model 

c:\Users\HuyenDT\source\repos\sttproject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("facebook/wav2vec2-base")
tokenizer # tokenizer nào cũng na ná nhau thôi, dùng cái này có sẵn đã có decoder trong đó rồi 

Wav2Vec2CTCTokenizer(name_or_path='facebook/wav2vec2-base', vocab_size=32, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'word_delimiter_token': '|'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
	1: AddedToken("<s>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
	2: AddedToken("</s>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
	3: AddedToken("<unk>", rstrip=True, lstrip=True, single_word=False, normalized=False, special=False),
	4: AddedToken("|", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
tokenizer.vocab
# <pad>: blank token for repeating characters vd: hh.ee.ll.-.lo => colapse into hello

In [ ]:
class LibrispeechDataset(Dataset):
    def __init__(self, 
                 path_to_data_root, 
                 include_splits = ["dev-other"], # ["train-clean-100", "train-clean-360", "train-other-500"], 
                 sampling_rate = 16000, 
                 num_audio_channels = 1): 
        
        self.sampling_rate = sampling_rate 
        self.num_audio_channels = num_audio_channels

        if isinstance(include_splits, str): # if it is a string 
            include_splits = [include_splits] # to make sure include_splits is a list, even if it is a string => 1 item
        
        self.librispeech_data = []
        for s in include_splits: 
            path_to_split = os.path.join(path_to_data_root, s)  # format: speaker/section/audio
            
            for speaker in os.listdir(path_to_split): 
                path_to_speaker = os.path.join(path_to_split, speaker)
                # print(speaker)

                for section in os.listdir(path_to_speaker): 
                    path_to_section = os.path.join(path_to_speaker, section)
                    files = os.listdir(path_to_section)

                    transcript_file = [path for path in files if ".txt" in path][0]
                    with open(os.path.join(path_to_section, transcript_file), "r") as f: 
                        transcripts = f.readlines()

                    for line in transcripts: 
                        split_line = line.split() # default is space => return an array
                        audio_root = split_line[0]
                        audio_file = audio_root + ".flac"
                        full_path_to_audio_file = os.path.join(path_to_section, audio_file)
                        transcript = " ".join(split_line[1:]).strip()

                        self.librispeech_data.append(
                            (full_path_to_audio_file, transcript)
                        )
        # print(len(self.librispeech_data))
        # Create a transform to transfrom the audio waveform → Mel Spectrogram.
        # Waveform (1D signal) => STFT (Fourier Transform) -> Mel scaling -> Mel Spectrogram (2D tensor)
        # Mel scale: based on how people can hear the voice to separate into level (is log(Hz))
        self.audio2mels = T.MelSpectrogram(
            sampling_rate = sampling_rate, # tấn suất lấy mẫu của audio: esim: 16000 Hz = 1 giây có 16000 mẫu
            n_mels=80 # Mel filter banks: (optimal value) Output sẽ có 80 hàng (80 tần số mel), => chia trục tần số thành 80 dải Mel.
            # 40: lost data, 128-256: heavy, more RAM, and training time
        )

data = LibrispeechDataset("C:/Users/HuyenDT/Downloads/LibriSpeech")


2864


In [ ]:
os.listdir("C:/Users/HuyenDT/Downloads/LibriSpeech")